# XMCD Visualiser
### *{{beamline}}: {{date}}*
This notebook allows the user to interactively choose pairs of XAS measurements to subtract as XMCD measurements, using different backgrounds, then average these to generate a final XMCD pattern.

{{description}}

*mmg_toolbox* contains some useful tools for analysis of XAS spectra, in particular for calculation of subtracted
polarised spectra for calculation of dichroic signals like XMCD and XMLD.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

import numpy as np
import matplotlib.pyplot as plt
from mmg_toolbox import Experiment, module_info, xas
from mmg_toolbox.plotting.matplotlib import set_plot_defaults

print(module_info())
set_plot_defaults()  # set nice defaults for matplotlib plots

## Load Spectra


In [ ]:
# Create experiment object - monitors one or more data folders for files
exp = Experiment('{{experiment_dir}}', instrument='{{beamline}}')

print(exp)

# print all scans in directory (could take a while...)
# print(exp.all_scans_str())

In [ ]:
# pick scan range
scan_range = {{scan_numbers}}
sample_name=  'mysample'

# Load scan data and plot raw spectra
spectras = exp.load_xas(*scan_range, sample_name=sample_name)  # only loads NXxas spectra (energy scans) and creates a Spectra object

for xas_scan in spectras:
    print(xas_scan)


## Choose Pairs

In [ ]:
# Choose Pairs (choose one of the options below to define the pairs)
# 1. Automatic choice
pol_pairs = xas.polarised_pairs(*spectras)

# 2. Manual choice
# scan_numbers = [
#     # (pol1, pol2)
#     (37436, 37438),
#     (37437, 37439),
#
# ]
# pol_pairs = [
#     exp.load_xas(s1, s2, sample_name=sample_name, mode=mode, match_metadata=False)
#     for s1, s2 in scan_numbers
# ]

# show table
s = f'| {pol_pairs[0][0].metadata.pol} | {pol_pairs[0][1].metadata.pol} |\n| --- | --- |\n'
for p1_scan, p2_scan in pol_pairs:
    s += f"| {p1_scan.metadata.scan_no} | {p2_scan.metadata.scan_no} |\n"
display(Markdown(s))

## Select Spectra & Average

In [ ]:
%matplotlib widget

modes = list(pol_pairs[0][0].spectra)
backgrounds = ['None'] + list(xas.spectra.BACKGROUND_FUNCTIONS)

dropdown1 = widgets.Dropdown(options=modes, value=modes[0], description='Measurement Mode')
dropdown2 = widgets.Dropdown(options=backgrounds, value='None', description='Background Function')
figs = [plt.subplots() for s1, s2 in pol_pairs]
av_fig, av_ax = plt.subplots()
checks = [widgets.Checkbox(value=True) for s1, s2 in pol_pairs]
pol1_list: list[xas.SpectraContainer] = []
pol2_list: list[xas.SpectraContainer] = []

def plot(change=None):
    mode = dropdown1.value
    background = dropdown2.value
    pol1_list.clear()
    pol2_list.clear()

    for (scan1, scan2), (fig, ax), check in zip(pol_pairs, figs, checks):
        ax.clear()
        scan1_proc = scan1.divide_by_preedge()
        scan2_proc = scan2.divide_by_preedge()
        if background != 'None':
            scan1_proc = scan1_proc.remove_background(background)
            scan2_proc = scan2_proc.remove_background(background)
        subtract = scan1_proc - scan2_proc
        if check.value:
            pol1_list.append(scan1_proc)
            pol2_list.append(scan2_proc)
        subtract.create_combined_axes(mode, ax)

    av_ax.clear()
    pol1 = container_functions.average_scans(*pol1_list)
    pol2 = container_functions.average_scans(*pol2_list)
    av_subtract = pol1 - pol2
    av_subtract.create_combined_axes(mode, av_ax)
    av_ax.set_title('Average ' + av_subtract.label())

for widget in [dropdown1, dropdown2] + [check for check in checks]:
    widget.observe(plot, names="value")

plot()
display(widgets.VBox(
    [widgets.HBox([dropdown1, dropdown2])] +
    [widgets.HBox([fig.canvas, check]) for (fig, ax), check in zip(figs, checks)] +
    [av_fig.canvas]
))

In [ ]:
mode = dropdown1.value
pol1 = container_functions.average_scans(*pol1_list)
pol2 = container_functions.average_scans(*pol2_list)
av_subtract = pol1 - pol2
fig, ax = plt.subplots()
ax = av_subtract.create_combined_axes(mode, ax)
ax.set_title('Average ' + av_subtract.label())
plt.show()

In [ ]:
# create processed nexus file
output_filename = f"{av_subtract.name}.nxs"
av_subtract.write_nexus(output_filename)

In [ ]:
# Plot NeXus data
from mmg_toolbox import data_file_reader

scan = data_file_reader(output_filename)
scan.plot()
plt.show()

In [ ]:
# Create ASCII data table
output_filename = f"{av_subtract.label()}.csv"
av_subtract.write_csv(output_filename, mode)

print(f"\nFile '{output_filename}':")
with open(output_filename, 'r') as f:
    print(f.read())

In [ ]:
# Plot ASCII data
data = np.loadtxt(output_filename, delimiter=',')
fig, ax = plt.subplots()
ax.plot(data[:, 0], data[:, 1])
ax.set_xlabel('E [eV]')
ax.set_ylabel('XMCD')
ax.set_title(output_filename)
plt.show()